In [35]:
from pyspark.sql import SparkSession
import spark

In [2]:
spark = SparkSession.builder \
        .appName("Customer Data Processing") \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 23:45:18 INFO SparkEnv: Registering MapOutputTracker
26/04/16 23:45:18 INFO SparkEnv: Registering BlockManagerMaster
26/04/16 23:45:18 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/16 23:45:18 INFO SparkEnv: Registering OutputCommitCoordinator


## Read and Process Customers Data

In [3]:
!hadoop fs -ls /data/

#!hadoop fs -ls /tmp/

Found 6 items
-rw-r--r--   2 root hadoop          0 2026-04-16 23:41 /data/_SUCCESS
-rw-r--r--   2 root hadoop    1060750 2026-04-16 23:42 /data/customers.csv
-rw-r--r--   2 root hadoop   10528211 2026-04-16 23:42 /data/customers_10mb.csv
-rw-r--r--   2 root hadoop  168541068 2026-04-16 23:43 /data/customers_150mb.csv
-rw-r--r--   2 root hadoop     863301 2026-04-16 23:42 /data/orders.csv
-rw-r--r--   2 root hadoop     266416 2026-04-16 23:41 /data/part-00000-224c25a6-71b4-4f63-8c2f-8eaaab53ef29-c000.snappy.parquet


In [4]:
# Read the file from HDFS

df = spark.read \
      .format("csv") \
      .option("header", "true") \
      .load("/data/customers.csv")

In [5]:
spark

In [6]:
df.count()

17653

In [7]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [9]:
# AS I am not enhancing the schema, I will get the column type as string

df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: string (nullable = true)



#### Changing column dataypes

In [10]:
# Changing the datatypes of registration_date from string to date and is_active from string to boolean

from pyspark.sql.functions import *

df = df.withColumn('registration_date', to_date(col('registration_date'), 'yyyy-MM-dd')) \
        .withColumn('is_active', col('is_active').cast('boolean'))

In [11]:
# Checking for the changed datatypes

df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



#### Data Preprocessing and Exploratory Data Analysis on customers Data

In [12]:
# If we have Unknown data in city/state/country, then fill it with fillna

df = df.fillna({'city': 'Unknow', 'state': 'Unknow', 'country': 'Unknow'})

In [13]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [14]:
# Extracting the year and month from registration_date

df = df.withColumn('registration_year', year('registration_date')) \
       .withColumn('registration_month', month('registration_date'))

In [15]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|customer_id|      name|     city|      state|country|registration_date|is_active|registration_year|registration_month|
+-----------+----------+---------+-----------+-------+-----------------+---------+-----------------+------------------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    false|             2023|                 6|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|             2023|                12|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|             2023|                10|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    false|             2023|                10|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    false|             2023|                 3|
+-----------+----------+---------+------

In [16]:
# Unique cities, states, country

unique_cities = df.select(countDistinct('city'))
unique_cities.show()

unique_states = df.select(countDistinct('state'))
unique_states.show()

unique_country = df.select(countDistinct('country')).collect()
print("Unique_Country:", unique_country[0][0])

+--------------------+
|count(DISTINCT city)|
+--------------------+
|                   8|
+--------------------+



+---------------------+
|count(DISTINCT state)|
+---------------------+
|                    7|
+---------------------+

Unique_Country: 1


In [17]:
# GROUPBY by city

df.groupBy('city').count().orderBy(col('count').desc()).show()

+---------+-----+
|     city|count|
+---------+-----+
|     Pune| 2243|
|Hyderabad| 2242|
|  Kolkata| 2223|
|Bangalore| 2211|
|    Delhi| 2200|
|Ahmedabad| 2198|
|  Chennai| 2194|
|   Mumbai| 2142|
+---------+-----+



In [18]:
# GROUPBY by state and country

df.groupBy('state', 'country').count().orderBy(col('count').desc()).show()

+-----------+-------+-----+
|      state|country|count|
+-----------+-------+-----+
|      Delhi|  India| 2578|
|    Gujarat|  India| 2543|
| Tamil Nadu|  India| 2536|
|  Telangana|  India| 2520|
|West Bengal|  India| 2503|
|Maharashtra|  India| 2490|
|  Karnataka|  India| 2483|
+-----------+-------+-----+



In [19]:
# Pivot Table - Count of Active and Inactive Users Per state

df.groupBy('state').pivot('is_Active').count().show()

+-----------+-----+----+
|      state|false|true|
+-----------+-----+----+
|    Gujarat| 1211|1332|
|      Delhi| 1356|1222|
|  Karnataka| 1207|1276|
|  Telangana| 1294|1226|
|Maharashtra| 1260|1230|
| Tamil Nadu| 1284|1252|
|West Bengal| 1306|1197|
+-----------+-----+----+



#### Window Functions and Ranking Analysis

In [20]:
## Window function performs calculations across a group of rows related to current row without reducing the number of rows.It adds new column while keeping all original rows.
from pyspark.sql.window import Window

window_spec = Window.partitionBy('state').orderBy(col('registration_date').desc())

##### 
1. rank(): Same values -> same rank. Skips numbers (gaps)
2. dense_rank(): Same values → same rank. No gaps
3. row_number(): Gives unique sequential numbers, No duplicates

In [21]:
df = df.withColumn('rank', rank().over(window_spec)) \
       .withColumn('dense_rank', dense_rank().over(window_spec)) \
       .withColumn('row_number', row_number().over(window_spec))

In [22]:
df.show(5)

+-----------+--------------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+
|customer_id|          name|     city|state|country|registration_date|is_active|registration_year|registration_month|rank|dense_rank|row_number|
+-----------+--------------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+
|         61|   Customer_61|Hyderabad|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|         1|         1|
|        501|  Customer_501|   Mumbai|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|         1|         2|
|       2763| Customer_2763|     Pune|Delhi|  India|       2023-12-31|     true|             2023|                12|   1|         1|         3|
|      12858|Customer_12858|Ahmedabad|Delhi|  India|       2023-12-31|     true|             2023|                12|   1|        

In [23]:
df.select('name', 'city', 'state', 'registration_date', 'rank', 'dense_rank', 'row_number').show(10)

+--------------+---------+-----+-----------------+----+----------+----------+
|          name|     city|state|registration_date|rank|dense_rank|row_number|
+--------------+---------+-----+-----------------+----+----------+----------+
|   Customer_61|Hyderabad|Delhi|       2023-12-31|   1|         1|         1|
|  Customer_501|   Mumbai|Delhi|       2023-12-31|   1|         1|         2|
| Customer_2763|     Pune|Delhi|       2023-12-31|   1|         1|         3|
|Customer_12858|Ahmedabad|Delhi|       2023-12-31|   1|         1|         4|
|Customer_13570|Bangalore|Delhi|       2023-12-31|   1|         1|         5|
|Customer_14970|  Chennai|Delhi|       2023-12-31|   1|         1|         6|
|Customer_16447|Ahmedabad|Delhi|       2023-12-31|   1|         1|         7|
|Customer_16709|  Chennai|Delhi|       2023-12-31|   1|         1|         8|
|Customer_17129|  Chennai|Delhi|       2023-12-31|   1|         1|         9|
|  Customer_250|     Pune|Delhi|       2023-12-30|  10|         

In [24]:
# Filtering

df_recent_customers = df.filter(col('registration_date') >= '2023-07-01')
df_recent_customers.count()

9025

In [25]:
# Oldest and newest customer per city

df.groupBy('city').agg(min('registration_date').alias('oldest'), max('registration_date').alias('newest')).show()

+---------+----------+----------+
|     city|    oldest|    newest|
+---------+----------+----------+
|    Delhi|2023-01-01|2023-12-31|
|  Kolkata|2023-01-01|2023-12-31|
|Hyderabad|2023-01-01|2023-12-31|
|Bangalore|2023-01-01|2023-12-31|
|Ahmedabad|2023-01-01|2023-12-31|
|  Chennai|2023-01-01|2023-12-31|
|   Mumbai|2023-01-01|2023-12-31|
|     Pune|2023-01-01|2023-12-31|
+---------+----------+----------+



In [26]:
output_path = '/data/'
#df.write.mode('overwrite').parquet(output_path)  ### this overwrites all the files and creates the parquet file

In [27]:
## Wrote to a HDFS /data/

!hadoop fs -ls /data/
# !hadoop fs -ls /tmp/

Found 6 items
-rw-r--r--   2 root hadoop          0 2026-04-16 23:41 /data/_SUCCESS
-rw-r--r--   2 root hadoop    1060750 2026-04-16 23:42 /data/customers.csv
-rw-r--r--   2 root hadoop   10528211 2026-04-16 23:42 /data/customers_10mb.csv
-rw-r--r--   2 root hadoop  168541068 2026-04-16 23:43 /data/customers_150mb.csv
-rw-r--r--   2 root hadoop     863301 2026-04-16 23:42 /data/orders.csv
-rw-r--r--   2 root hadoop     266416 2026-04-16 23:41 /data/part-00000-224c25a6-71b4-4f63-8c2f-8eaaab53ef29-c000.snappy.parquet


## Read and Process Orders Data

In [32]:
orders_df = spark.read \
    .format("csv") \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .load("/data/orders.csv")

In [33]:
orders_df.show(10)

+--------+-----------+----------+------------------+---------+
|order_id|customer_id|order_date|      total_amount|   status|
+--------+-----------+----------+------------------+---------+
|       0|       3692|2024-09-03| 547.7160076008001|  Shipped|
|       1|      11055|2024-08-10| 577.8942599188381|  Pending|
|       2|       6963|2024-08-22| 484.2085562764487|  Pending|
|       3|      13268|2024-09-01| 366.3286882431848|Cancelled|
|       4|       1131|2024-08-09| 896.9588380686909|  Pending|
|       5|      15211|2024-05-03|486.30584827618145|  Shipped|
|       6|       3209|2024-08-24| 800.9795667933956|  Shipped|
|       7|      15964|2024-11-24| 641.3521833737843|Cancelled|
|       8|       9697|2024-06-18| 789.1864475667901|Delivered|
|       9|       8917|2024-09-01| 878.8901928499223|  Shipped|
+--------+-----------+----------+------------------+---------+
only showing top 10 rows



In [42]:
orders_df.count()

17653

#### Data Preprocessing and Exploratory Data Analysis on Orders Data

In [30]:
# This is the ouput when I am not using the inferSchema : datatype is strings for all columns
# orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- status: string (nullable = true)



In [34]:
# This is the output when using inferSchema = true: Spark automatically assigns column data types based on the data.
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- status: string (nullable = true)



In [37]:
# Handling missing values
orders_df = orders_df.fillna({'status' : 'Unknown'})

In [38]:
orders_df.show(6)

+--------+-----------+----------+------------------+---------+
|order_id|customer_id|order_date|      total_amount|   status|
+--------+-----------+----------+------------------+---------+
|       0|       3692|2024-09-03| 547.7160076008001|  Shipped|
|       1|      11055|2024-08-10| 577.8942599188381|  Pending|
|       2|       6963|2024-08-22| 484.2085562764487|  Pending|
|       3|      13268|2024-09-01| 366.3286882431848|Cancelled|
|       4|       1131|2024-08-09| 896.9588380686909|  Pending|
|       5|      15211|2024-05-03|486.30584827618145|  Shipped|
+--------+-----------+----------+------------------+---------+
only showing top 6 rows



In [39]:
# Extracting the year and month from order date

orders_df = orders_df.withColumn('order_year', year('order_date')) \
       .withColumn('order_month', month('order_date'))

In [40]:
orders_df.show(5)

+--------+-----------+----------+-----------------+---------+----------+-----------+
|order_id|customer_id|order_date|     total_amount|   status|order_year|order_month|
+--------+-----------+----------+-----------------+---------+----------+-----------+
|       0|       3692|2024-09-03|547.7160076008001|  Shipped|      2024|          9|
|       1|      11055|2024-08-10|577.8942599188381|  Pending|      2024|          8|
|       2|       6963|2024-08-22|484.2085562764487|  Pending|      2024|          8|
|       3|      13268|2024-09-01|366.3286882431848|Cancelled|      2024|          9|
|       4|       1131|2024-08-09|896.9588380686909|  Pending|      2024|          8|
+--------+-----------+----------+-----------------+---------+----------+-----------+
only showing top 5 rows



In [41]:
# Unique customer_id

unique_customer_id = orders_df.select(countDistinct('customer_id'))
unique_customer_id.show()

+---------------------------+
|count(DISTINCT customer_id)|
+---------------------------+
|                      11138|
+---------------------------+



In [43]:
# GROUPBY by status

orders_df.groupBy('status').count().orderBy(col('count').desc()).show()

+---------+-----+
|   status|count|
+---------+-----+
|Cancelled| 4469|
|  Pending| 4457|
|  Shipped| 4386|
|Delivered| 4341|
+---------+-----+



In [44]:
### Min and Max order_date

orders_df.groupBy('status').agg(min('order_date').alias('old_order'), max('order_date').alias('new_order')).show()

+---------+----------+----------+
|   status| old_order| new_order|
+---------+----------+----------+
|Cancelled|2024-01-01|2024-12-30|
|Delivered|2024-01-01|2024-12-30|
|  Shipped|2024-01-01|2024-12-30|
|  Pending|2024-01-01|2024-12-30|
+---------+----------+----------+



### Joining and Analyzing Customer Data and Order Data

In [70]:
# Join

customers_order_df = df.join(orders_df, 
                             'customer_id', 
                             how = 'inner')

In [71]:
customers_order_df.count()

17653

In [72]:
customers_order_df.show(5)

+-----------+------------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+--------+----------+------------------+---------+----------+-----------+
|customer_id|        name|     city|state|country|registration_date|is_active|registration_year|registration_month|rank|dense_rank|row_number|order_id|order_date|      total_amount|   status|order_year|order_month|
+-----------+------------+---------+-----+-------+-----------------+---------+-----------------+------------------+----+----------+----------+--------+----------+------------------+---------+----------+-----------+
|         61| Customer_61|Hyderabad|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|         1|         1|   17283|2024-01-21|171.91694722937515|  Pending|      2024|          1|
|         61| Customer_61|Hyderabad|Delhi|  India|       2023-12-31|    false|             2023|                12|   1|         1|         

In [73]:
# can also convert the data to pandas and have a look at it
#customers_order_df.limit(5).toPandas()

,customer_id,name,city,state,country,registration_date,is_active,registration_year,registration_month,rank,dense_rank,row_number,order_id,order_date,total_amount,status,order_year,order_month
0,61,Customer_61,Hyderabad,Delhi,India,2023-12-31,False,2023,12,1,1,1,17283,2024-01-21,171.916947,Pending,2024,1
1,61,Customer_61,Hyderabad,Delhi,India,2023-12-31,False,2023,12,1,1,1,17058,2024-08-25,605.956729,Pending,2024,8
2,61,Customer_61,Hyderabad,Delhi,India,2023-12-31,False,2023,12,1,1,1,13508,2024-05-29,492.687016,Pending,2024,5
3,61,Customer_61,Hyderabad,Delhi,India,2023-12-31,False,2023,12,1,1,1,2610,2024-07-09,535.024579,Delivered,2024,7
4,501,Customer_501,Mumbai,Delhi,India,2023-12-31,False,2023,12,1,1,2,16373,2024-10-29,944.536265,Pending,2024,10


In [74]:
# Total orders per customer

customers_orders_count = customers_order_df.groupBy('customer_id').count().orderBy(col('count').desc())
customers_orders_count.show(5)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|      11776|    7|
|       3884|    6|
|       7566|    6|
|       3243|    6|
|      13034|    6|
+-----------+-----+
only showing top 5 rows



In [75]:
# Total spend per customer

customer_orders_spent = customers_order_df.groupBy('customer_id').agg(sum('total_amount').alias('total_spend'))
customer_orders_spent.show(6)

+-----------+------------------+
|customer_id|       total_spend|
+-----------+------------------+
|        296| 758.4097164274607|
|      13192| 2226.750452156505|
|      16250| 519.9827815079817|
|       9583|393.86551623411617|
|      13610|  94.6021305392804|
|       1512|1315.7938279538707|
+-----------+------------------+
only showing top 6 rows



In [76]:
# Total spend per customer, looking for highest spend customer

customer_orders_spent = customers_order_df.groupBy('customer_id').agg(sum('total_amount').alias('total_spend')).orderBy(col('total_spend').desc())
customer_orders_spent.show(6)

+-----------+------------------+
|customer_id|       total_spend|
+-----------+------------------+
|       3336| 4362.550733141537|
|       3884|  4187.99763145619|
|      16020|3967.2692112582276|
|      14372| 3961.787139557334|
|      14933|3828.5841072418348|
|       7566| 3647.119115720654|
+-----------+------------------+
only showing top 6 rows



In [77]:
# Average spend per customer

customer_avg_spent = customers_order_df.groupBy('customer_id').agg(avg('total_amount').alias('avg_spend')).orderBy(col('avg_spend').desc())
customer_avg_spent.show(6)

+-----------+-----------------+
|customer_id|        avg_spend|
+-----------+-----------------+
|      11854| 999.864258397557|
|         46| 999.592553819927|
|      17590|999.5726342625253|
|      11587|999.5595016039513|
|       6816|999.4348902885968|
|       9648| 999.421469440536|
+-----------+-----------------+
only showing top 6 rows



In [78]:
# Order by status

orders_status_count = customers_order_df.groupBy('status').count().orderBy(col('count').desc())
orders_status_count.show()

+---------+-----+
|   status|count|
+---------+-----+
|Cancelled| 4469|
|  Pending| 4457|
|  Shipped| 4386|
|Delivered| 4341|
+---------+-----+



In [79]:
# order by month

order_by_month = customers_order_df.groupBy('order_month').count().orderBy(col('order_month'))
order_by_month.show()


+-----------+-----+
|order_month|count|
+-----------+-----+
|          1| 1499|
|          2| 1368|
|          3| 1539|
|          4| 1457|
|          5| 1518|
|          6| 1455|
|          7| 1517|
|          8| 1472|
|          9| 1446|
|         10| 1513|
|         11| 1426|
|         12| 1443|
+-----------+-----+



In [84]:
# Window Function
#from pyspark.sql.window import Window

window_spec = Window.orderBy(col('total_spend').desc())

In [87]:
ranked_customers = customer_orders_spent.withColumn('rank', rank().over(window_spec)) \
                                        .withColumn('dense_rank', dense_rank().over(window_spec))
ranked_customers.show(6)

26/04/17 14:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:36:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+-----------+------------------+----+----------+
|customer_id|       total_spend|rank|dense_rank|
+-----------+------------------+----+----------+
|       3336| 4362.550733141537|   1|         1|
|       3884|  4187.99763145619|   2|         2|
|      16020|3967.2692112582276|   3|         3|
|      14372| 3961.787139557334|   4|         4|
|      14933|3828.5841072418348|   5|         5|
|       7566| 3647.119115720654|   6|         6|
+-----------+------------------+----+----------+
only showing top 6 rows



In [88]:
customer_orders_spent.printSchema() , customers_orders_count.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- total_spend: double (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- count: long (nullable = false)



(None, None)

In [89]:
# Finding Customers with High order Frequency(from customers_orders_count) but Low Total Spend (from customer_orders_spent)

customer_spend_vs_orders = customers_orders_count.join(customer_orders_spent,
                                                      'customer_id',
                                                      how = 'inner')

In [92]:
customer_spend_vs_orders.orderBy(col('count').desc(), col('total_spend')).show(6)

+-----------+-----+------------------+
|customer_id|count|       total_spend|
+-----------+-----+------------------+
|      11776|    7|  3438.36692751212|
|       5160|    6| 1656.737343311546|
|       4294|    6| 1821.603928366352|
|       3243|    6|2860.1827303387754|
|      14838|    6| 2894.355602564058|
|      13034|    6|3195.0224622202268|
+-----------+-----+------------------+
only showing top 6 rows



In [ ]:
output_path = '/data/'
customers_orders_df.write.mode('overwrite').parquet(output_path)  ### this overwrites all the files and creates the parquet file